# Fine-tune yolov8s-oiv7 on a balanced subset of Open Images V7

**Run this notebook in Google Colab, not locally** (your laptop has no CUDA GPU, which makes this impractical there).

1. Open [colab.research.google.com](https://colab.research.google.com), upload this file.
2. `Runtime -> Change runtime type -> T4 GPU`.
3. Run all cells top to bottom.

**What this does:** downloads `SAMPLES_PER_CLASS_TRAIN` training images and `SAMPLES_PER_CLASS_VAL` validation images for *each* of the 601 Open Images classes already baked into `yolov8s-oiv7.pt` (so the class list matches your app exactly), fine-tunes the pretrained model on that balanced set, then exports the result to ONNX in the same format `export_model.py` already produces.

**Time estimate:** with the defaults below (60 train + 15 val images/class = ~45,000 images), expect roughly 1-2 hours for the download and 1-3 hours for training on a free T4, depending on queue load. This fits inside a single free Colab session (~12h) but can disconnect if idle — keep the tab open. Start with small values first (e.g. `SAMPLES_PER_CLASS_TRAIN = 10`) to sanity-check the whole pipeline in ~15 minutes before committing to a full run.

In [ ]:
!pip install -q ultralytics fiftyone onnx onnxslim

In [ ]:
# ---- Tune these ----
SAMPLES_PER_CLASS_TRAIN = 60   # start with ~10 for a quick smoke test
SAMPLES_PER_CLASS_VAL = 15
EPOCHS = 30
IMGSZ = 640
BATCH = 16
DATASET_DIR = "/content/dataset"
RUNS_DIR = "/content/runs"

In [ ]:
from ultralytics import YOLO

# Loads the same pretrained weights export_model.py uses. We fine-tune from
# here instead of training from scratch, and we read its class list so the
# fine-tuned model keeps the exact same 601 classes/order your app expects.
base_model = YOLO("yolov8s-oiv7.pt")
class_names = [base_model.names[i] for i in range(len(base_model.names))]
print(len(class_names), "classes")

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

def download_balanced_split(split, samples_per_class):
    """Downloads a small, per-class-capped slice of Open Images V7 and merges
    it into one FiftyOne dataset, so rare classes aren't drowned out by
    common ones (plain random sampling would be dominated by Person, etc.)."""
    merged = fo.Dataset(f"oiv7_{split}_merged", overwrite=True)
    for i, cls in enumerate(class_names):
        try:
            tmp = foz.load_zoo_dataset(
                "open-images-v7",
                split=split,
                label_types=["detections"],
                classes=[cls],
                max_samples=samples_per_class,
                shuffle=True,
                seed=51,
                dataset_name=f"oiv7_{split}_{i}",
            )
            merged.merge_samples(tmp)
            fo.delete_dataset(tmp.name)
        except Exception as e:
            print(f"  skip [{cls}]: {e}")
        if i % 25 == 0:
            print(f"{split}: {i}/{len(class_names)} classes processed, {len(merged)} samples so far")
    return merged

train_set = download_balanced_split("train", SAMPLES_PER_CLASS_TRAIN)
val_set = download_balanced_split("validation", SAMPLES_PER_CLASS_VAL)
print("train samples:", len(train_set), "val samples:", len(val_set))

In [ ]:
# Converts the FiftyOne datasets into the image+label folder layout YOLO expects.
train_set.export(
    export_dir=DATASET_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    split="train",
    classes=class_names,
    label_field="detections",
)
val_set.export(
    export_dir=DATASET_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    split="val",
    classes=class_names,
    label_field="detections",
)
print("Exported to", DATASET_DIR)

In [ ]:
# Fine-tune: starts from the already-trained weights (not from scratch) and
# keeps adjusting them on this new, more balanced sample of the same classes.
model = YOLO("yolov8s-oiv7.pt")
results = model.train(
    data=f"{DATASET_DIR}/dataset.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=10,
    project=RUNS_DIR,
    name="finetune",
)

In [ ]:
import json, os, shutil

# Same export settings export_model.py uses, so the output drops straight
# into your project's models/ folder with no other changes needed.
best = YOLO(f"{RUNS_DIR}/finetune/weights/best.pt")
onnx_path = best.export(format="onnx", imgsz=IMGSZ, nms=False, simplify=True, opset=17)

os.makedirs("/content/models", exist_ok=True)
shutil.copy(onnx_path, "/content/models/yolov8s-oiv7.onnx")

labels = [best.names[i] for i in range(len(best.names))]
with open("/content/models/labels.json", "w") as f:
    json.dump(labels, f, indent=0)

print("Saved /content/models/yolov8s-oiv7.onnx and labels.json with", len(labels), "classes")

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("/content/models_finetuned", "zip", "/content/models")
files.download("/content/models_finetuned.zip")

## After downloading

1. Unzip `models_finetuned.zip`.
2. Replace the files in your project's `models/` folder:
   - `models/yolov8s-oiv7.onnx`
   - `models/labels.json`
3. Restart your local server (`python -m http.server 8080`) and reload `http://localhost:8080` — the app loads the new weights with no code changes.
4. Compare against the README's informal accuracy test to see if it improved.